In [116]:
from dash import Dash, dcc, html, Input, Output
import pandas as pd
import plotly.graph_objs as go
import dash_bootstrap_components as dbc
import numpy as np
import folium
from folium import plugins 
from jupyter_dash import JupyterDash
from math import sin, cos
import plotly.graph_objects as go
import plotly.express as px


In [117]:
# Carregamento do dataset
df_imovel = pd.read_csv('../../Data/Processed/imovel_tratado.csv', sep=';', encoding='utf-8', low_memory=False)
df_populacao = pd.read_excel('../../Data/Raw/populacao.xls', skiprows=1)

In [118]:
df_populacao.columns 


Index(['BRASIL E UNIDADES DA FEDERAÇÃO', 'POPULAÇÃO ESTIMADA', 'Unnamed: 2'], dtype='object')

In [119]:
def mapaLoja(path_excel):
    """
    Gera um mapa Folium das lojas e retorna o HTML do mapa
    """
    df = pd.read_excel(path_excel, engine='openpyxl')
    df_clean = df.dropna(subset=['Latitude', 'Longitude'])

    cidades_unicas = df_clean['CIDADE'].unique()
    cores = [
        'red','blue','green','purple','orange','darkred',
        'lightred','beige','darkblue','darkgreen','cadetblue',
        'darkpurple','pink','lightblue','lightgreen','gray',
        'black','lightgray'
    ]
    cor_por_cidade = {cidade: cores[i % len(cores)] for i, cidade in enumerate(cidades_unicas)}

    mapa = folium.Map(location=[-15.7801, -47.9292], zoom_start=4, tiles='OpenStreetMap')

    for _, row in df_clean.iterrows():
        popup_html = f"""
        <div style="font-family: Arial; width: 300px;">
            <h4 style="margin-bottom: 10px; color: #2c3e50;">{row['HUB']}</h4>
            <hr style="margin: 5px 0;">
            <table style="width: 100%; font-size: 12px;">
                <tr><td><b>Empresa:</b></td><td>{row['EMPRESA']}</td></tr>
                <tr><td><b>Cidade:</b></td><td>{row['CIDADE']}</td></tr>
                <tr><td><b>Capital:</b></td><td>{row['CAPITAL']}</td></tr>
                <tr><td><b>Região:</b></td><td>{row['REGIÃO']}</td></tr>
                <tr><td><b>UF:</b></td><td>{row['UF']}</td></tr>
                <tr><td><b>CEP:</b></td><td>{row['CEP']}</td></tr>
                <tr><td><b>Endereço:</b></td><td>{row['ENDEREÇO ATUAL']}</td></tr>
                <tr><td><b>Acessibilidade:</b></td><td>{row['Acessibilidade']}</td></tr>
                <tr><td><b>Supervisão:</b></td><td>{row['SUPERVISÃO']}</td></tr>
                <tr><td><b>E-mail:</b></td><td>{row['E-MAIL SUPERVISÃO']}</td></tr>
                <tr><td><b>Telefone:</b></td><td>{row['TELEFONE LOJA']}</td></tr>
            </table>
        </div>
        """
        folium.Marker(
            location=[row['Latitude'], row['Longitude']],
            popup=folium.Popup(popup_html, max_width=350),
            tooltip=f"🏪 {row['HUB']} - {row['CIDADE']}/{row['UF']}",
            icon=folium.Icon(color=cor_por_cidade[row['CIDADE']], icon='shopping-cart', prefix='fa')
        ).add_to(mapa)

    title_html = '''
    <div style="position: fixed; 
                top: 10px; left: 50%;
                transform: translateX(-50%);
                background-color: white; border: 2px solid grey;
                border-radius: 5px; padding: 10px; font-weight: bold;
                z-index: 9999;">
        Distribuição de Lojas SOLDI no Brasil
    </div>
    '''
    mapa.get_root().html.add_child(folium.Element(title_html))

    legenda_html = '<div style="position: fixed; bottom: 50px; left: 50px; background-color: white; border: 2px solid grey; border-radius: 5px; z-index:9999; padding:10px;">'
    legenda_html += '<h4>Cidades</h4>'
    for cidade, cor in list(cor_por_cidade.items())[:10]:
        legenda_html += f'<p><span style="color: {cor};">●</span> {cidade}</p>'
    legenda_html += '</div>'
    mapa.get_root().html.add_child(folium.Element(legenda_html))

    plugins.Fullscreen(title='Tela Cheia').add_to(mapa)
    plugins.MiniMap(toggle_display=True).add_to(mapa)
    plugins.MeasureControl(primary_length_unit='kilometers').add_to(mapa)

    return mapa._repr_html_()

In [120]:
mapa_lojahtml = mapaLoja("../../Data/Raw/endereco_lojas_2025.xlsx")

In [122]:
# Dados
x = np.linspace(0, 10, 100)
y1 = np.sin(x)
y2 = np.cos(x)

# Cria os gráficos Plotly e exporta para HTML
fig_seno = go.Figure()
fig_seno.add_trace(go.Scatter(x=x, y=y1, mode="lines", name="Seno", line=dict(color="cyan")))
fig_seno.update_layout(title="Função Seno", template="plotly_dark")
seno_html = fig_seno.to_html(include_plotlyjs="cdn", full_html=False)

fig_cos = go.Figure()
fig_cos.add_trace(go.Scatter(x=x, y=y2, mode="lines", name="Cosseno", line=dict(color="orange")))
fig_cos.update_layout(title="Função Cosseno", template="plotly_dark")
cos_html = fig_cos.to_html(include_plotlyjs=False, full_html=False)


In [ ]:
dashboard_html = f"""
<!DOCTYPE html>
<html lang="pt-BR">
<head>
  <meta charset="UTF-8">
  <title>Dashboard Moderno</title>
  <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>

  <style>
    :root {{
      --bg-color: #111;
      --sidebar-bg: #1e1e1e;
      --text-color: #fff;
      --accent-color: #0d6efd;
    }}

    body.light-theme {{
      --bg-color: #f5f5f5;
      --sidebar-bg: #ffffff;
      --text-color: #222;
      --accent-color: #111;
    }}

    body {{
      margin: 0;
      font-family: 'Segoe UI', Roboto, Arial, sans-serif;
      background-color: var(--bg-color);
      color: var(--text-color);
      overflow-x: hidden;
      transition: background-color 0.4s, color 0.4s;
    }}

    /* ===== Sidebar ===== */
    #sidebar {{
      position: fixed;
      top: 0;
      left: -220px;
      bottom: 0;
      width: 220px;
      background-color: var(--sidebar-bg);
      box-shadow: 2px 0 10px rgba(0,0,0,0.3);
      transition: left 0.3s ease;
      padding-top: 60px;
      z-index: 1000;
      border-right: 1px solid rgba(255,255,255,0.1);
    }}

    #sidebar.open {{
      left: 0;
    }}

    #menu-label {{
      font-size: 22px;
      font-weight: 600;
      color: var(--accent-color);
      margin: 0 0 20px 25px;
      display: block;
    }}

    #sidebar a {{
      display: block;
      padding: 12px 25px;
      text-decoration: none;
      color: var(--text-color);
      opacity: 0.8;
      transition: background-color 0.3s ease, opacity 0.3s;
      border-radius: 6px;
      margin: 5px 15px;
    }}

    #sidebar a:hover {{
      background-color: var(--accent-color);
      color: #fff;
      opacity: 1;
    }}

    /* ===== Botão de MENU ===== */
    #menu-btn {{
      position: fixed;
      top: 100%;
      left: 15px;
      width: 45px;
      height: 45px;
      font-size: 22px;
      cursor: pointer;
      border: 2px solid transparent;
      background-color: transparent;
      color: var(--accent-color);
      border-radius: 50%;
      z-index: 1001;
      display: flex;
      align-items: center;
      justify-content: center;
      transition: background-color 0.4s ease, color 0.4s ease, transform 0.4s ease;
    }}

    #menu-btn:hover {{
      background-color: var(--accent-color);
      color: #fff;
      transform: scale(1.05);
    }}

    .sidebar-open #menu-btn {{
      transform: rotate(360deg);
    }}

    /* ===== Conteúdo ===== */
    #content {{
      padding: 40px 30px;
      margin-left: 0;
      transition: margin-left 0.3s ease;
    }}

    .sidebar-open #content {{
      margin-left: 220px;
    }}

    /* ===== BOTÃO DE TEMA (preto/branco) ===== */
    #theme-toggle {{
      position: fixed;
      top: 20px;
      right: 20px;
      width: 40px;
      height: 40px;
      border-radius: 50%;
      border: 2px solid var(--accent-color);
      background-color: transparent;
      color: var(--accent-color);
      cursor: pointer;
      font-size: 18px;
      display: flex;
      align-items: center;
      justify-content: center;
      transition: background-color 0.4s, color 0.4s, transform 0.3s;
      z-index: 1100;
    }}

    #theme-toggle:hover {{
      background-color: var(--accent-color);
      color: #fff;
      transform: scale(1.1);
    }}

    /* ===== Responsividade ===== */
    @media (max-width: 700px) {{
      .sidebar-open #content {{
        margin-left: 0;
      }}
      #menu-btn {{
        top: 30px;
      }}
    }}
  </style>
</head>

<body>
  <button id="menu-btn" onclick="toggleSidebar()">&#62;</button>
  <button id="theme-toggle" onclick="toggleTheme()">🌙</button>

  <div id="sidebar">
    <span id="menu-label">Dashboard</span>
    <a href="#" onclick="showTab('seno')">Busca Hub</a>
    <a href="#" onclick="showTab('cos')">Gráfico Estados</a>
    <a href="#" onclick="showTab('mapaloja')">Mapa Lojas</a>
    <a href="#" onclick="showTab('config')">Configurações</a>
    <a href="#" onclick="showTab('mapaEstados')">Mapa de Estados</a>
  </div>

  <div id="content">
    <div id="seno" style="display:block;">{seno_html}</div>
    <div id="cos" style="display:none;">{cos_html}</div>
    <div id="mapaloja" style="display:none;">{mapa_lojahtml}</div> 
    <div id="mapaEstados" style="display:none;">{mapaEstados}</div>
    <div id="config" style="display:none;">
      <h3>Configurações</h3>
      <p>Aqui você pode adicionar controles para o gráfico futuramente.</p>
    </div>
  </div>

  <script>
    function toggleSidebar() {{
      const body = document.body;
      const sidebar = document.getElementById('sidebar');
      const btn = document.getElementById('menu-btn');
      body.classList.toggle('sidebar-open');
      sidebar.classList.toggle('open');
      btn.textContent = sidebar.classList.contains('open') ? '<' : '>';
    }}

    // 🔹 versão que funciona pra todas as abas
    function showTab(tabName) {{
      document.querySelectorAll('#content > div').forEach(div => {{
        div.style.display = 'none';
      }});
      document.getElementById(tabName).style.display = 'block';
      if (window.innerWidth <= 700) toggleSidebar();
    }}

    function toggleTheme() {{
      document.body.classList.toggle('light-theme');
      const themeBtn = document.getElementById('theme-toggle');
      const isLight = document.body.classList.contains('light-theme');
      themeBtn.textContent = isLight ? '💡' : '🌙';
    }}
  </script>
</body>
</html>
"""
with open("dashboard.html", "w", encoding="utf-8") as f:
    f.write(dashboard_html)

print(" Arquivo 'dashboard.html' criado")


 Arquivo 'dashboard.html' criado
